# Decision Tree Classifier

A tree-based classification algorithm that makes predictions by recursively partitioning the feature space using feature-based conditions. The algorithm builds a binary tree structure where each internal node represents a decision based on a feature value, and each leaf node represents a class prediction.

<br>

<p align="center">
<img src="visualizations/decision_tree.png" width="600">
</p>

## Mathematical Foundation

**Goal:** Build a tree that partitions the feature space to minimize impurity in each resulting region.

**Impurity Measures:**

* **Gini impurity:**
$$
Gini = 1 - \sum_{k=1}^{K} p_k^2
$$

* **Entropy:**
$$
Entropy = -\sum_{k=1}^{K} p_k \log_2(p_k)
$$

* **Information Gain:**
$$
IG = I(parent) - \sum_{j} \frac{n_j}{n} I(child_j)
$$

Where $p_k$ is the proportion of samples belonging to class $k$ in a node.

## Algorithm Steps

1. **Initialize:** Begin with all training data at the root node

2. **Node Splitting:** For each node, evaluate all possible splits:
   - **Continuous features:** Consider splits of the form $x_i \leq t$
   - **Categorical features:** Evaluate partitions of categories

3. **Best Split Selection:** Choose the split that maximizes impurity reduction (information gain)

4. **Recursive Partitioning:** 
   - Partition data according to the chosen split
   - Recursively apply the process to each child node

5. **Stopping Criteria:** Stop when:
   - Node is pure (all samples have same class)
   - Maximum depth is reached
   - Minimum samples per split threshold is met
   - No split improves impurity significantly

6. **Prediction:** Classify new samples by traversing from root to leaf, following the learned decision rules

## Key Characteristics

### Advantages
* Highly interpretable structure (can visualize decision paths)
* Handles both numerical and categorical features naturally
* No assumptions about data distributions
* No need for feature scaling or normalization
* Automatically performs feature selection
* Can capture non-linear relationships

### Limitations
* Prone to overfitting, especially with deep trees
* High variance (small changes in data can create very different trees)
* Biased toward features with more levels
* Greedy algorithm may not find globally optimal tree
* Can create overly complex trees that don't generalize well

### When to Use
* When interpretability is crucial
* For exploratory data analysis and feature understanding
* With mixed data types (numerical and categorical)
* When you need to explain individual predictions
* As a baseline model before trying ensemble methods

In [1]:
import numpy as np
from tqdm import tqdm
from cifar10.cifar10_utils import get_data, get_all_data, get_test_data, extract_images_pca, normalize_data

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from images.image_preprocessing import extract_raw_pixels, extract_color_histogram, extract_hog, extract_lbp

In [2]:
class Node:
    # Represents a node in the decision tree
    def __init__(
        self, feature_index=None, threshold=None, left=None, right=None, *, value=None
    ):
        self.feature_index = feature_index  # Feature used for split
        self.threshold = threshold  # Threshold value for the split
        self.left = left  # Left child
        self.right = right  # Right child
        self.value = value  # Class label for leaf nodes

    def is_leaf(self):
        return self.value is not None  # True if this is a leaf node


class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.node_count = 0

    def fit(self, X, y):
        self.root = self._build_tree(X, y)  # Start building the tree from root

    def _entropy(self, y):
        # Computes entropy of label distribution
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        return -np.sum(probs * np.log2(probs + 1e-9))  # small epsilon to avoid log(0)

    def _best_split(self, X, y):
        # Finds the best feature and threshold to split on using information gain
        best_gain = -1
        split_idx, split_thresh = None, None
        current_entropy = self._entropy(y)

        n_samples, n_features = X.shape
        num_features_to_try = min(50, n_features)
        feature_indices = np.random.choice(
            n_features, num_features_to_try, replace=False
        )

        for i, feature in enumerate(feature_indices):
            feature_values = X[:, feature]
            thresholds = np.percentile(
                feature_values, np.linspace(0, 100, 10)
            )  # try only 10 thresholds per feature

            for t in thresholds:
                left_idx = X[:, feature] < t
                right_idx = X[:, feature] >= t
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue

                # Calculate information gain
                left_entropy = self._entropy(y[left_idx])
                right_entropy = self._entropy(y[right_idx])
                p = len(y[left_idx]) / len(y)
                gain = current_entropy - (p * left_entropy + (1 - p) * right_entropy)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_thresh = t

        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        # Recursively builds the tree
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        self.node_count += 1

        if (
            depth >= self.max_depth
            or n_labels == 1
            or n_samples < self.min_samples_split
        ):
            # Stop condition: return a leaf node
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idx, threshold = self._best_split(X, y)
        if feat_idx is None:
            return Node(value=self._most_common_label(y))

        # Split the dataset
        left_idxs = X[:, feat_idx] < threshold
        right_idxs = X[:, feat_idx] >= threshold

        left = self._build_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs], y[right_idxs], depth + 1)
        return Node(feature_index=feat_idx, threshold=threshold, left=left, right=right)

    def _most_common_label(self, y):
        # Returns the most frequent class label
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

    def _traverse(self, x, node):
        # Traverses the tree for a single sample
        if node.is_leaf():
            return node.value
        if x[node.feature_index] < node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def predict(self, X):
        # Predicts labels for a batch of samples
        return np.array([self._traverse(x, self.root) for x in X])

In [3]:
# 1) Load data
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

# Preprocess: extract HOG features, apply PCA and normalize
x_train_pca, pca = extract_images_pca(extract_hog(x_train))
x_train_norm, mean, std = normalize_data(x_train_pca)
x_test_norm = normalize_data(pca.transform(extract_hog(x_test)), mean, std)

# Train algorithm
tree = DecisionTree(max_depth=10)
tree.fit(x_train_norm, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = tree.predict(x_test_norm)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Test accuracy: 0.3535


# Random Forest

Ensemble of decision trees, combines the predictions of multiple trees to improve generalization and reduce overfitting.

<br>

<p align="center">
<img src="visualizations/random_forest.png" width="1000">
</p>

## Mathematical Foundation

**Core Idea:** Train multiple decision trees on different random subsets of the data and features. Combine their predictions (e.g., by majority vote) to obtain the final output.

**Bootstrap Aggregating (Bagging):** Each tree is trained on a random sample (with replacement) of the training data.

**Feature Randomness:** At each split, a random subset of features is considered—introducing more diversity among trees.

## Algorithm Steps

1. **For each tree** $t = 1, 2, \ldots, T$:
2. **Bootstrap Sampling:** Create bootstrap sample by sampling $n$ examples with replacement
3. **Build Decision Tree:** Train tree on bootstrap sample with feature randomness at each split
4. **Store Tree:** Add tree to ensemble

**Prediction:** Each tree outputs a class label; the forest returns the **majority vote** across all trees.

## Key Characteristics

### Advantages
* Reduces overfitting compared to single decision trees
* Handles missing values and maintains accuracy for large datasets
* Provides feature importance estimates
* Robust to outliers and noise
* Works well with mixed data types (numerical and categorical)
* Parallelizable training

### Limitations
* Less interpretable than single decision trees
* Can overfit with very noisy data
* Memory intensive for large forests
* Biased towards categorical variables with many categories
* Not optimal for linear relationships

### When to Use
* When you need robust performance with minimal tuning
* For datasets with mixed feature types
* When feature importance information is valuable
* For problems where interpretability is less critical
* When you have sufficient computational resources

In [2]:
from sklearn.tree import DecisionTreeClassifier
from collections import Counter

In [3]:
class RandomForest:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        # Train each tree on a bootstrap sample of the data
        self.trees = []
        n_samples = X.shape[0]

        for _ in tqdm(range(self.n_estimators), desc="Training trees"):
            indices = np.random.choice(n_samples, n_samples, replace=True)  # Bootstrap sampling
            X_sample = X[indices]
            y_sample = y[indices]

            tree = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        # Collect predictions from all trees
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)  # shape: (n_samples, n_trees)

        # Majority vote across trees
        y_pred = np.array([Counter(row).most_common(1)[0][0] for row in tree_preds])
        return y_pred


In [6]:
# 1) Load data
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

# Preprocess: extract HOG features, apply PCA and normalize
x_train_pca, pca = extract_images_pca(extract_hog(x_train))
x_train_norm, mean, std = normalize_data(x_train_pca)
x_test_norm = normalize_data(pca.transform(extract_hog(x_test)), mean, std)

# Train algorithm
forest = RandomForest(n_estimators=50, max_depth=10)
forest.fit(x_train_norm, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = forest.predict(x_test_norm)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Training trees: 100%|██████████| 50/50 [01:30<00:00,  1.82s/it]

Test accuracy: 0.4269
